# Edge TTS — Arabic mapping v2 (MP3 only)

Round 4: promote comparison winners to **chosen**; generate final-consonant variants for **ض ق ص** only.

TTS input is **only** the variant text (plus `.` / `!` where listed).

Outputs: `generated_audio/edge_tts_mapping_v2/chosen/` and `.../comparison/`.


# Edge TTS — Arabic mapping v2 (MP3 only)

Round 4: promote comparison winners to **chosen**; generate final-consonant variants for **ض ق ص** only.

TTS input is **only** the variant text (plus `.` / `!` where listed).

Outputs: `generated_audio/edge_tts_mapping_v2/chosen/` and `.../comparison/`.


## 1. Dependencies

In [1]:
import subprocess
import sys

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
        print(f"[deps OK] {pkg}")
        return True
    except ImportError:
        print(f"[deps] installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)
        print(f"[deps OK] {pkg} (installed)")
        return True

_ensure("edge-tts", "edge_tts")
_ensure("nest_asyncio")
_ensure("IPython", "IPython")
print("Edge TTS dependency bootstrap done.")


[deps OK] edge-tts
[deps OK] nest_asyncio
[deps OK] IPython
Edge TTS dependency bootstrap done.


## 2. Paths and output folders


In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
AUDIO_ROOT = BASE_DIR / "generated_audio"
MAPPING_ROOT = AUDIO_ROOT / "edge_tts_mapping_v2"
DIR_CHOSEN = MAPPING_ROOT / "chosen"
DIR_COMPARISON = MAPPING_ROOT / "comparison"

DIR_NUMBERS = AUDIO_ROOT / "edge_tts_numbers"
DIR_PHRASES = AUDIO_ROOT / "edge_tts_phrases"

for d in (DIR_CHOSEN, DIR_COMPARISON, DIR_NUMBERS, DIR_PHRASES):
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", BASE_DIR)
print("mapping v2:", MAPPING_ROOT)
print("chosen:", DIR_CHOSEN)
print("comparison:", DIR_COMPARISON)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
mapping v2: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2
chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
comparison: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison


## 3. Egyptian letter, number, and phrase mappings

In [3]:
# Chosen: (stem, letter, pronunciation without period)
CHOSEN_MAPPINGS = [
    ("letter_ا_alif", "ا", "أَلِف"),
    ("letter_ت_teh", "ت", "تِه"),
    ("comp_seh_ث_ث_ه", "ث", "ثِه"),
    ("letter_ج_geem", "ج", "جِيم"),
    ("letter_ح_ha", "ح", "حَا"),
    ("letter_خ_kha", "خ", "خَه"),
    ("letter_ر_ra", "ر", "رَا"),
    ("letter_س_seen", "س", "سِين"),
    ("letter_ش_sheen", "ش", "شِين"),
    ("letter_ظ_dha", "ظ", "ظَه"),
    ("letter_ع_ein", "ع", "عِين"),
    ("letter_غ_ghein", "غ", "غِين"),
    ("letter_م_meem", "م", "مِيم"),
    ("letter_ن_noon", "ن", "نُون"),
    ("letter_ي_yeh", "ي", "يِه"),
    ("comp_ta_ط_ط_ه", "ط", "طَه"),
    ("comp_faa_ف_ف_ه", "ف", "فِه"),
    ("comp_ha_ه_ه_ه", "ه", "هِه"),
]

# Copy comparison winners -> chosen (first existing source wins)
COPY_TO_CHOSEN = [
    ("comp_dal_د_دال_bang", ["comp_dal_د_dal_latin"]),
    ("comp_kaf_ك_كاف_sukoon", []),
    ("comp_lam_ل_lam_latin", []),
    ("comp_waw_و_واو_diac", []),
    ("comp_zal_ذ_ذال_bang", []),
    ("comp_zay_ز_zeen_latin_long", []),
]

# Round 4: clearer final consonant + stopped ending (ض ق ص only)
FINAL_COMPARISON_VARIANTS = {
    "ض": [
        ("ضادْ", "final_d_sukoon"),
        ("ضادْ!", "final_d_bang"),
        ("ضادْـ", "final_d_tatweel"),
        ("ضاد", "final_d_plain"),
        ("ضَا دْ", "final_d_split_diac"),
        ("ضا دْ", "final_d_split"),
        ("ضَادْ", "final_d_diac"),
        ("daadْ", "final_d_latin_long_sukoon"),
        ("dadْ", "final_d_latin_sukoon"),
        ("daad", "final_d_latin_long"),
        ("dad", "final_d_latin"),
    ],
    "ق": [
        ("قافْ", "final_f_sukoon"),
        ("قافْ!", "final_f_bang"),
        ("قافْـ", "final_f_tatweel"),
        ("قاف", "final_f_plain"),
        ("قَا فْ", "final_f_split_diac"),
        ("قا فْ", "final_f_split"),
        ("قَافْ", "final_f_diac"),
        ("qaafْ", "final_f_latin_long_sukoon"),
        ("qafْ", "final_f_latin_sukoon"),
        ("qaaf", "final_f_latin_long"),
        ("qaf", "final_f_latin"),
    ],
    "ص": [
        ("صادْ", "final_d_sukoon"),
        ("صادْ!", "final_d_bang"),
        ("صادْـ", "final_d_tatweel"),
        ("صاد", "final_d_plain"),
        ("صَا دْ", "final_d_split_diac"),
        ("صا دْ", "final_d_split"),
        ("صَادْ", "final_d_diac"),
        ("saadْ", "final_d_latin_long_sukoon"),
        ("sadْ", "final_d_latin_sukoon"),
        ("saad", "final_d_latin_long"),
        ("sad", "final_d_latin"),
    ],
}

COMP_FILE_PREFIX = {"ض": "dad", "ق": "qaf", "ص": "sad", "د": "dal", "ك": "kaf", "ل": "lam", "و": "waw", "ذ": "zal", "ز": "zay"}

CHOSEN_SKIP_REGEN = frozenset({"comp_faa_ف_ف_ه", "comp_ha_ه_ه_ه"})

NUMBER_PRON = {
    0: "صِفْر", 1: "وَاحِد", 2: "اِتْنِين", 3: "تَلَاتَه", 4: "أَرْبَعَه",
    5: "خَمْسَه", 6: "سِتَّه", 7: "سَبْعَه", 8: "تَمَانْيَه", 9: "تِسْعَه", 10: "عَشَرَه",
}
PHRASES = [
    ("ana_basme3ak", "أَنَا بَسْمَعَك."),
    ("mama", "مَامَا."),
    ("beit", "بَيْت."),
    ("khamsa", "خَمْسَه."),
    ("itneen", "اِتْنِين."),
    ("ahlan_beek", "أَهْلًا بِيك."),
    ("ezzayak", "إِزَّيَّك."),
    ("ayez_arooh_elbeit", "عَايِز أَرُوح البَيْت."),
]

def with_period(text: str) -> str:
    t = text.strip()
    if t.endswith((".", "!", "?")):
        return t
    return t + "."

def number_tts(n: int) -> str:
    return with_period(NUMBER_PRON[n])

def _base_arabic_chars(text: str) -> list[str]:
    out = []
    for c in text:
        if c in "._" or c.isspace():
            continue
        if "\u0621" <= c <= "\u064a" or c in "أإآء":
            out.append(c)
    return out

def variant_path_part(pron: str) -> str:
    raw = pron.strip().strip(".!?")
    latin = raw.replace("ْ", "")
    if latin.isascii() and latin.isalpha():
        return latin.lower()
    return "".join(_base_arabic_chars(raw))

def comparison_stem(letter: str, pron: str, tag: str) -> str:
    prefix = COMP_FILE_PREFIX[letter]
    part = variant_path_part(pron) or tag
    return f"comp_{prefix}_{letter}_{part}_{tag}"

FINAL_COMP_TOTAL = sum(len(v) for v in FINAL_COMPARISON_VARIANTS.values())
print(f"Chosen mappings: {len(CHOSEN_MAPPINGS)}")
print(f"Copy to chosen: {len(COPY_TO_CHOSEN)}")
print(f"Final comparison letters: {''.join(FINAL_COMPARISON_VARIANTS)} ({FINAL_COMP_TOTAL} variants)")



Chosen mappings: 18
Copy to chosen: 6
Final comparison letters: ضقص (33 variants)


## 6b. Copy comparison winners to chosen


In [4]:
import shutil

COPY_CHOSEN_LOG = []
print("=== Copy comparison winners -> chosen ===")
for chosen_stem, fallbacks in COPY_TO_CHOSEN:
    sources = [chosen_stem, *fallbacks]
    src_path = None
    used_stem = None
    for stem in sources:
        candidate = DIR_COMPARISON / f"{stem}.mp3"
        if candidate.is_file():
            src_path = candidate
            used_stem = stem
            break
    if src_path is None:
        print(f"MISSING: none of {sources} found in comparison")
        print("---")
        continue
    dest = DIR_CHOSEN / f"{chosen_stem}.mp3"
    shutil.copy2(src_path, dest)
    action = "copied" if used_stem == chosen_stem else f"copied (from {used_stem})"
    print(f"chosen: {chosen_stem}.mp3 <- {src_path.name} ({action})")
    print(f"output MP3 path: {dest.resolve()}")
    COPY_CHOSEN_LOG.append({"chosen_stem": chosen_stem, "source": used_stem, "path": dest.resolve()})
    print("---")

print(f"Copied to chosen: {len(COPY_CHOSEN_LOG)}/{len(COPY_TO_CHOSEN)}")



=== Copy comparison winners -> chosen ===
chosen: comp_dal_د_دال_bang.mp3 <- comp_dal_د_دال_bang.mp3 (copied)
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_dal_د_دال_bang.mp3
---
chosen: comp_kaf_ك_كاف_sukoon.mp3 <- comp_kaf_ك_كاف_sukoon.mp3 (copied)
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_kaf_ك_كاف_sukoon.mp3
---
chosen: comp_lam_ل_lam_latin.mp3 <- comp_lam_ل_lam_latin.mp3 (copied)
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_lam_ل_lam_latin.mp3
---
chosen: comp_waw_و_واو_diac.mp3 <- comp_waw_و_واو_diac.mp3 (copied)
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_waw_و_واو_diac.mp3
---
chosen: comp_zal_ذ_ذال_bang.mp3 <- comp_zal_ذ_ذال_bang.mp3 (copied)
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2

## 4. Select Edge TTS voice

In [5]:
import asyncio

import edge_tts
import nest_asyncio

nest_asyncio.apply()

PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
EDGE_VOICE = None
EDGE_TTS_LOADED = False
_ALL_VOICES = None

async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES

async def select_edge_voice() -> str:
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}

    for pref in PREFERRED_VOICES:
        if pref in by_short:
            return pref

    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    print("Preferred voices unavailable. Available Arabic voices:")
    for name in ar_names:
        print(" ", name)

    eg = [
        v["ShortName"]
        for v in voices
        if "EG" in v.get("Locale", "") or "-EG-" in v.get("ShortName", "")
    ]
    if eg:
        return eg[0]
    if ar_names:
        return ar_names[0]
    raise RuntimeError("No Arabic Edge TTS voice found")

EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
EDGE_TTS_LOADED = True
print(f"Selected Edge TTS voice: {EDGE_VOICE}")


Selected Edge TTS voice: ar-EG-SalmaNeural


## 5. Edge TTS synthesis (MP3 only)


In [6]:
import re
import time
from typing import Optional

from IPython.display import Audio as IPA, display

CHOSEN_LOG = []
COMPARISON_LOG = []
PLAYBACK_SHOWN = False

def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"

async def synth_edge_mp3(text: str, mp3_path: Path) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    await comm.save(str(mp3_path))
    return time.perf_counter() - t0

async def generate_mp3(
    letter: str,
    pron: str,
    mp3_path: Path,
    show_playback: bool = False,
) -> dict | None:
    global PLAYBACK_SHOWN
    tts_text = with_period(pron)
    infer_s = await synth_edge_mp3(tts_text, mp3_path)
    if not mp3_path.is_file():
        print("FAILED:", mp3_path)
        return None
    entry = {
        "letter": letter,
        "pron": pron,
        "tts_text": tts_text,
        "mp3_path": mp3_path.resolve(),
        "infer_s": infer_s,
        "bytes": mp3_path.stat().st_size,
    }
    print(f"letter: {letter}")
    print(f"variant text sent to Edge TTS: {tts_text}")
    print(f"output MP3 path: {entry['mp3_path']}")
    print(f"inference: {infer_s:.3f}s | {format_bytes(entry['bytes'])}")
    print("---")
    if show_playback:
        display(IPA(filename=str(mp3_path)))
        PLAYBACK_SHOWN = True
    return entry

def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


## 6. Generate chosen mappings (MP3)


In [7]:
import shutil

CHOSEN_LOG.clear()
CHOSEN_OK = 0

print("=== Chosen mappings ===")
for stem, letter, pron in CHOSEN_MAPPINGS:
    mp3_path = DIR_CHOSEN / f"{stem}.mp3"
    comp_path = DIR_COMPARISON / f"{stem}.mp3"
    tts_text = with_period(pron)

    if mp3_path.is_file() and stem in CHOSEN_SKIP_REGEN:
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (kept, skipped regen)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    if stem in CHOSEN_SKIP_REGEN and comp_path.is_file() and not mp3_path.is_file():
        shutil.copy2(comp_path, mp3_path)
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (copied from comparison)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    if mp3_path.is_file():
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (kept)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    entry = run_async(generate_mp3(letter, pron, mp3_path, show_playback=False))
    if entry:
        entry["stem"] = stem
        CHOSEN_LOG.append(entry)
        CHOSEN_OK += 1

print(f"Chosen: {CHOSEN_OK}/{len(CHOSEN_MAPPINGS)} MP3 files")


=== Chosen mappings ===
letter: ا
variant text sent to Edge TTS: أَلِف.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ا_alif.mp3 (kept)
---
letter: ت
variant text sent to Edge TTS: تِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ت_teh.mp3 (kept)
---
letter: ث
variant text sent to Edge TTS: ثِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_seh_ث_ث_ه.mp3 (kept)
---
letter: ج
variant text sent to Edge TTS: جِيم.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ج_geem.mp3 (kept)
---
letter: ح
variant text sent to Edge TTS: حَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ح_ha.mp3 (kept)
---
letter: خ
variant text sent to Edge TTS: خَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Mod

## 7. Final-consonant comparison variants (ض ق ص only)

Clear prior round-4 clips for these letters only. MP3 only. TTS input is **only** the variant text.



In [8]:
removed = 0
for letter in FINAL_COMPARISON_VARIANTS:
    prefix = COMP_FILE_PREFIX[letter]
    for old in DIR_COMPARISON.glob(f"comp_{prefix}_{letter}_*_final_*.mp3"):
        old.unlink()
        removed += 1
print(f"Cleared {removed} prior round-4 comparison MP3(s) for ض/ق/ص")

COMPARISON_LOG.clear()
COMP_OK = 0

for letter, variants in FINAL_COMPARISON_VARIANTS.items():
    print(f"\n=== {letter} variants ===")
    for pron, tag in variants:
        stem = comparison_stem(letter, pron, tag)
        mp3_path = DIR_COMPARISON / f"{stem}.mp3"
        entry = run_async(generate_mp3(letter, pron, mp3_path, show_playback=False))
        if entry:
            entry["stem"] = stem
            entry["tag"] = tag
            COMPARISON_LOG.append(entry)
            COMP_OK += 1

COMPARISON_OK = COMP_OK == FINAL_COMP_TOTAL and COMP_OK > 0
print(f"\nComparison (ض/ق/ص round 4): {COMP_OK}/{FINAL_COMP_TOTAL} MP3 files")



Cleared 33 prior round-4 comparison MP3(s) for ض/ق/ص

=== ض variants ===


letter: ض
variant text sent to Edge TTS: ضادْ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_sukoon.mp3
inference: 0.596s | 7.88 KB
---


letter: ض
variant text sent to Edge TTS: ضادْ!
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_bang.mp3
inference: 0.579s | 7.59 KB
---


letter: ض
variant text sent to Edge TTS: ضادْـ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضادـ_final_d_tatweel.mp3
inference: 0.708s | 7.88 KB
---


letter: ض
variant text sent to Edge TTS: ضاد.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_plain.mp3
inference: 0.571s | 7.88 KB
---


...

letter: ص
variant text sent to Edge TTS: صَادْ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_صاد_final_d_diac.mp3
inference: 0.638s | 7.88 KB
---


letter: ص
variant text sent to Edge TTS: saadْ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_saad_final_d_latin_long_sukoon.mp3
inference: 0.781s | 7.88 KB
---


letter: ص
variant text sent to Edge TTS: sadْ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_sad_final_d_latin_sukoon.mp3
inference: 0.782s | 8.30 KB
---


letter: ص
variant text sent to Edge TTS: saad.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_saad_final_d_latin_long.mp3
inference: 0.557s | 7.88 KB
---


letter: ص
variant text sent to Edge TTS: sad.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_sad_final_d_latin.mp3
inference: 0.579s | 8.30 KB
---

Comparison (ض/ق/ص round 4): 33/33 MP3 files


## 8. Playback — new comparison variants


In [9]:
def _play_entry(entry: dict, title: str = ""):
    if title:
        print(title)
    print(f"  letter: {entry['letter']}")
    print(f"  variant text sent to Edge TTS: {entry['tts_text']}")
    print(f"  output MP3 path: {entry['mp3_path']}")
    display(IPA(filename=str(entry["mp3_path"])))
    print()

by_letter = {}
for e in COMPARISON_LOG:
    by_letter.setdefault(e["letter"], []).append(e)

print("=== Final-consonant comparison variants (ض / ق / ص) ===")
for letter in FINAL_COMPARISON_VARIANTS:
    print(f"--- {letter} variants ---")
    for entry in by_letter.get(letter, []):
        _play_entry(entry)

PLAYBACK_SHOWN = True



=== Final-consonant comparison variants (ض / ق / ص) ===
--- ض variants ---
  letter: ض
  variant text sent to Edge TTS: ضادْ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_sukoon.mp3



  letter: ض
  variant text sent to Edge TTS: ضادْ!
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_bang.mp3



  letter: ض
  variant text sent to Edge TTS: ضادْـ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضادـ_final_d_tatweel.mp3



  letter: ض
  variant text sent to Edge TTS: ضاد.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_plain.mp3



  letter: ض
  variant text sent to Edge TTS: ضَا دْ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dad_ض_ضاد_final_d_split_diac.mp3


...


  letter: ص
  variant text sent to Edge TTS: صَادْ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_صاد_final_d_diac.mp3



  letter: ص
  variant text sent to Edge TTS: saadْ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_saad_final_d_latin_long_sukoon.mp3



  letter: ص
  variant text sent to Edge TTS: sadْ.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_sad_final_d_latin_sukoon.mp3



  letter: ص
  variant text sent to Edge TTS: saad.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_saad_final_d_latin_long.mp3



  letter: ص
  variant text sent to Edge TTS: sad.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_sad_ص_sad_final_d_latin.mp3


## 9. Number tests (MP3, optional)


In [10]:
NUMBERS_LOG = []
ok_num = 0

for n in range(0, 11):
    tts_text = number_tts(n)
    mp3_path = DIR_NUMBERS / f"num_{n:02d}.mp3"
    entry = run_async(generate_mp3(str(n), NUMBER_PRON[n], mp3_path))
    if entry:
        NUMBERS_LOG.append(entry)
        ok_num += 1

NUMBERS_OK = ok_num == 11
print(f"Numbers: {ok_num}/11 MP3 files")


letter: 0
variant text sent to Edge TTS: صِفْر.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_00.mp3
inference: 0.673s | 8.30 KB
---


letter: 1
variant text sent to Edge TTS: وَاحِد.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_01.mp3
inference: 0.574s | 9.00 KB
---


letter: 2
variant text sent to Edge TTS: اِتْنِين.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_02.mp3
inference: 0.593s | 9.98 KB
---


letter: 3
variant text sent to Edge TTS: تَلَاتَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_03.mp3
inference: 0.605s | 9.98 KB
---


letter: 4
variant text sent to Edge TTS: أَرْبَعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_04.mp3
inference: 0.617s | 9.70 KB
---


...

letter: 6
variant text sent to Edge TTS: سِتَّه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_06.mp3
inference: 0.580s | 9.42 KB
---


letter: 7
variant text sent to Edge TTS: سَبْعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_07.mp3
inference: 0.643s | 9.70 KB
---


letter: 8
variant text sent to Edge TTS: تَمَانْيَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_08.mp3
inference: 0.653s | 10.12 KB
---


letter: 9
variant text sent to Edge TTS: تِسْعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_09.mp3
inference: 0.626s | 9.42 KB
---


letter: 10
variant text sent to Edge TTS: عَشَرَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_10.mp3
inference: 0.620s | 10.27 KB
---
Numbers: 11/11 MP3 files


## 10. Egyptian Arabic phrase tests (MP3, optional)


In [11]:
PHRASES_LOG = []
ok_phrase = 0

for slug, phrase in PHRASES:
    tts_text = with_period(phrase)
    mp3_path = DIR_PHRASES / f"{slug}.mp3"
    entry = run_async(generate_mp3(slug, phrase.rstrip("."), mp3_path))
    if entry:
        PHRASES_LOG.append(entry)
        ok_phrase += 1

PHRASES_OK = ok_phrase == len(PHRASES)
print(f"Phrases: {ok_phrase}/{len(PHRASES)} MP3 files")


letter: ana_basme3ak
variant text sent to Edge TTS: أَنَا بَسْمَعَك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ana_basme3ak.mp3
inference: 0.620s | 11.67 KB
---


letter: mama
variant text sent to Edge TTS: مَامَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\mama.mp3
inference: 0.589s | 8.72 KB
---


letter: beit
variant text sent to Edge TTS: بَيْت.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\beit.mp3
inference: 0.605s | 8.02 KB
---


letter: khamsa
variant text sent to Edge TTS: خَمْسَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\khamsa.mp3
inference: 0.607s | 9.56 KB
---


letter: itneen
variant text sent to Edge TTS: اِتْنِين.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\itneen.mp3
inference: 0.633s | 9.98 KB
---


letter: ahlan_beek
variant text sent to Edge TTS: أَهْلًا بِيك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ahlan_beek.mp3
inference: 0.613s | 10.27 KB
---


letter: ezzayak
variant text sent to Edge TTS: إِزَّيَّك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ezzayak.mp3
inference: 0.625s | 10.12 KB
---


letter: ayez_arooh_elbeit
variant text sent to Edge TTS: عَايِز أَرُوح البَيْت.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ayez_arooh_elbeit.mp3
inference: 0.570s | 13.92 KB
---
Phrases: 8/8 MP3 files


## 11. Output summary


In [12]:
chosen_mp3 = sorted(DIR_CHOSEN.glob("*.mp3"))
comp_mp3 = sorted(DIR_COMPARISON.glob("*.mp3"))
round4_mp3 = sorted(DIR_COMPARISON.glob("*_final_*.mp3"))

print("Chosen folder:", DIR_CHOSEN)
for p in chosen_mp3:
    print(" ", p.name, format_bytes(p.stat().st_size))

print("\nComparison folder (all):", DIR_COMPARISON)
print(f"  total: {len(comp_mp3)} MP3 files")

print("\nRound-4 comparison (ض/ق/ص):")
for p in round4_mp3:
    print(" ", p.name, format_bytes(p.stat().st_size))

expected_chosen = len(CHOSEN_MAPPINGS) + len(COPY_TO_CHOSEN)
print(f"\nTotal chosen: {len(chosen_mp3)} (base {len(CHOSEN_MAPPINGS)} + promoted {len(COPY_TO_CHOSEN)} = {expected_chosen})")
print(f"Round-4 comparison: {len(round4_mp3)} (expected {FINAL_COMP_TOTAL})")



Chosen folder: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
  comp_dal_د_دال_bang.mp3 8.16 KB
  comp_faa_ف_ف_ه.mp3 7.31 KB
  comp_ha_ه_ه_ه.mp3 8.30 KB
  comp_kaf_ك_كاف_sukoon.mp3 7.59 KB
  comp_lam_ل_lam_latin.mp3 9.00 KB
  comp_seh_ث_ث_ه.mp3 7.17 KB
  comp_ta_ط_ط_ه.mp3 8.16 KB
  comp_waw_و_واو_diac.mp3 8.30 KB
  comp_zal_ذ_ذال_bang.mp3 8.16 KB
  comp_zay_ز_zeen_latin_long.mp3 8.44 KB
  letter_ا_alif.mp3 8.44 KB
  letter_ت_teh.mp3 8.02 KB
  letter_ج_geem.mp3 9.14 KB
  letter_ح_ha.mp3 7.88 KB
  letter_خ_kha.mp3 8.02 KB
  letter_ر_ra.mp3 7.73 KB
  letter_س_seen.mp3 9.28 KB
  letter_ش_sheen.mp3 9.42 KB
  letter_ظ_dha.mp3 8.44 KB
  letter_ع_ein.mp3 9.70 KB
  letter_غ_ghein.mp3 9.28 KB
  letter_م_meem.mp3 8.58 KB
  letter_ن_noon.mp3 8.58 KB
  letter_ي_yeh.mp3 7.45 KB

Comparison folder (all): C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison
  total: 100 MP3 files

Round-4 comparison (ض/ق/ص):
  comp_dad_ض_daad_fina

## 12. Final verification


In [13]:
checks = [
    ("Edge TTS loaded", EDGE_TTS_LOADED and bool(EDGE_VOICE)),
    ("Chosen MP3 generated", CHOSEN_OK == len(CHOSEN_MAPPINGS)),
    ("Promoted to chosen", len(COPY_CHOSEN_LOG) == len(COPY_TO_CHOSEN)),
    ("Round-4 comparison MP3", COMPARISON_OK),
    ("Playback displayed", PLAYBACK_SHOWN),
]

for label, ok in checks:
    mark = "OK" if ok else "FAIL"
    print(f"[{mark}] {label}")

ALL_OK = all(ok for _, ok in checks)
print("\nVoice:", EDGE_VOICE)
print("mapping v2:", MAPPING_ROOT)



[OK] Edge TTS loaded
[OK] Chosen MP3 generated
[OK] Promoted to chosen
[OK] Round-4 comparison MP3
[OK] Playback displayed

Voice: ar-EG-SalmaNeural
mapping v2: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2
